# LangGraph, Part 2 — Branching, Tools & Memory
### Module 6 — Stateful Agents (Part 2 of 2)

**Goal:** make the agent *decide* and *act*. We add **conditional edges** (the agent chooses its path), **tools** (it calls functions like a CVE lookup), and **memory** (it remembers across turns) — building a small **alert-triage agent**, close to Omar's `branching_conditional_logic.py` and `ethical_hacking_agent.py`.

> **Setup:** `pip install langgraph langchain-openai`. Guarded on `OPENAI_API_KEY`. Ships without outputs. Do Part 1 first.

## 🛠️ Setup

In [ ]:
# pip install langgraph langchain-openai python-dotenv
import os
try:
    from dotenv import load_dotenv, find_dotenv; load_dotenv(find_dotenv())
except Exception:
    pass
from typing import TypedDict
HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))

class TriageState(TypedDict):
    alert: str          # the incoming security alert
    risk: str           # "high" or "low" (set by the router)
    cve_info: str       # filled by a tool if we investigate
    report: str         # final summary
print("Setup ready. Key found:", HAS_KEY)

# 🔀 1 — Conditional Edges (the agent decides)

- A **router** node inspects the state and returns a label
- `add_conditional_edges` maps each label to the next node
- Example: high-risk alerts → investigate; low-risk → quick close
- This is how a graph branches instead of running straight through

> ### 🎤 Instructor Script
>
> The first new power is letting the agent choose its own path. We write a router function that looks at the state and returns a label — here, high or low risk based on the alert text. Then add_conditional_edges maps each label to a different next node: high-risk alerts go to an investigation step, low-risk ones go straight to a quick close. That branching — the graph deciding at runtime — is exactly what a plain chain can't do, and it mirrors Omar's severity-routing example.

In [ ]:
def classify(state: TriageState) -> dict:
    high = any(w in state["alert"].lower() for w in ["rce", "ransomware", "exfil", "critical", "admin"])
    risk = "high" if high else "low"
    print("  [node] classify -> risk =", risk)
    return {"risk": risk}

def route_by_risk(state: TriageState) -> str:
    return "investigate" if state["risk"] == "high" else "quick_close"   # label -> next node

# 🛠️ 2 — Give the Agent a Tool

- A tool is just a function the agent can call (e.g., CVE lookup)
- Here, the 'investigate' node calls a mock `lookup_cve` tool
- Real agents also use LangGraph's `ToolNode` for LLM-chosen tools
- Tools are how an agent affects the world beyond text

> ### 🎤 Instructor Script
>
> Next, tools. A tool is simply a function the agent can call to get information or take action — here a mock CVE lookup. Our investigate node calls it and stores the result in the state. In bigger agents you'd let the LLM decide which tool to call using LangGraph's ToolNode and tool-calling models, as in Omar's ethical-hacking agent; we keep it explicit here so the flow is clear. Tools are what turn a chatbot into an agent that does things.

In [ ]:
def lookup_cve(keyword: str) -> str:
    db = {"rce": "CVE-2026-0001: critical RCE; patch immediately.",
          "ransomware": "Multiple CVEs; isolate host and restore from backup."}
    for k, v in db.items():
        if k in keyword.lower():
            return v
    return "No specific CVE found; follow standard triage."

def investigate(state: TriageState) -> dict:
    info = lookup_cve(state["alert"])
    print("  [node] investigate -> tool returned:", info)
    summary = f"HIGH-RISK. {info}"
    if HAS_KEY:
        from langchain_openai import ChatOpenAI
        summary = ChatOpenAI(model="gpt-4o-mini", temperature=0).invoke(
            f"Write a 2-sentence triage report. Alert: {state['alert']}. CVE info: {info}").content
    return {"cve_info": info, "report": summary}

def quick_close(state: TriageState) -> dict:
    print("  [node] quick_close")
    return {"report": "LOW-RISK. Logged for monitoring; no immediate action."}

# 🔗 3 — Wire the Branching Graph

- Nodes: classify → (investigate | quick_close) → END
- `add_conditional_edges('classify', route_by_risk, {...})`
- Both branches lead to END
- Run it on a high-risk and a low-risk alert

> ### 🎤 Instructor Script
>
> Now we assemble the branching graph. After the classify node we add conditional edges: the router's label picks investigate or quick_close, and both lead to END. We compile and run it on two alerts — a critical RCE and a routine login — and watch the graph take different paths. Same graph, different journeys, decided by the data. That's stateful, conditional orchestration in a few lines.

In [ ]:
from langgraph.graph import StateGraph, END

b = StateGraph(TriageState)
b.add_node("classify", classify)
b.add_node("investigate", investigate)
b.add_node("quick_close", quick_close)
b.set_entry_point("classify")
b.add_conditional_edges("classify", route_by_risk,
                        {"investigate": "investigate", "quick_close": "quick_close"})
b.add_edge("investigate", END)
b.add_edge("quick_close", END)
triage = b.compile()

for alert in ["Critical RCE exploited on admin server", "Single failed login from a known device"]:
    print("ALERT:", alert)
    out = triage.invoke({"alert": alert, "risk": "", "cve_info": "", "report": ""})
    print("  REPORT:", out["report"], "\n")

# 🧠 4 — Add Memory (across turns)

- A **checkpointer** (`MemorySaver`) lets the graph remember state
- Each conversation gets a `thread_id`
- Re-invoking with the same thread continues where it left off
- Essential for multi-step, multi-turn agents

> ### 🎤 Instructor Script
>
> The last power is memory. By compiling the graph with a checkpointer — the simplest is MemorySaver — and invoking it with a thread id, the graph persists its state between calls. Same thread, and the agent picks up where it left off; new thread, and it starts fresh. This is what lets an agent hold a multi-step investigation or a conversation rather than treating every call as brand new. It's the foundation for the longer-running agents you'll orchestrate in the MCP module.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

triage_mem = b.compile(checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "incident-42"}}
out = triage_mem.invoke({"alert": "Ransomware detected on file server", "risk": "", "cve_info": "", "report": ""}, cfg)
print("Turn 1 report:", out["report"])
print("Saved state for thread 'incident-42' (re-invoke with the same thread_id to continue).")

# ✅ Summary — Part 2 & the Module

- **Conditional edges** → the agent chooses its path
- **Tools** → the agent acts (lookups, scans, APIs)
- **Memory** (checkpointer + thread_id) → multi-turn state
- You built a branching, tool-using, memory-backed triage agent
- → Mini-Project 4: your own LangGraph security agent

> ### 🎤 Instructor Script
>
> That completes LangGraph. You added conditional edges so the agent routes itself, tools so it can act, and memory so it can work across turns — the three ingredients of a real agent, all from Omar's examples. Put together, you built an alert-triage agent that classifies, branches, investigates with a tool, and remembers. Mini-Project 4 asks you to design your own LangGraph agent for a security task. Next module, MCP gives these agents a standard way to reach external tools and servers.

In [ ]:
print("Part 2 done: conditional edges, tools, and memory.")
print("Next: langgraph_sample.ipynb (template) and Mini-Project 4.")